<a href="https://colab.research.google.com/github/annatsamoyra-prog/data-story-/blob/main/newsbomb_scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [ ]:
BASE_URL = "https://www.newsbomb.gr"
CATEGORY_URL = "https://www.newsbomb.gr/technologia"

START_PAGE = 1
END_PAGE = 17

TARGET_YEAR = 2026
SOURCE_NAME = "Newsbomb"

MIN_DELAY = 1.0
MAX_DELAY = 2.0
REQUEST_TIMEOUT = 20


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

In [ ]:
def build_session():

    session = requests.Session()

    retries = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )

    adapter = HTTPAdapter(max_retries=retries)

    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update(HEADERS)

    return session


In [ ]:
def is_newsbomb_technology_article(url):

    if not url:
        return False

    return bool(
        re.search(
            r"newsbomb\.gr/technologia/story/\d+/",
            url
        )
    )



In [ ]:
def collect_article_urls(session):

    article_urls = set()

    for page in range(START_PAGE, END_PAGE + 1):

        url = f"{CATEGORY_URL}?page={page}"

        print(f"📄 Σελίδα {page}: {url}")

        try:

            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT
            )

            response.raise_for_status()

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            for a in soup.find_all("a", href=True):

                article_url = urljoin(
                    BASE_URL,
                    a["href"]
                )

                article_url = article_url.split("#")[0]

                if is_newsbomb_technology_article(article_url):

                    article_urls.add(article_url)

        except Exception as e:

            print(
                f"❌ Πρόβλημα στη σελίδα {page}: {e}"
            )

        time.sleep(
            random.uniform(MIN_DELAY, MAX_DELAY)
        )

    print(
        f"\n🔗 Βρέθηκαν {len(article_urls)} μοναδικά URLs."
    )

    return sorted(article_urls)


In [ ]:
MONTHS = {
    "Ιανουαρίου": 1,
    "Φεβρουαρίου": 2,
    "Μαρτίου": 3,
    "Απριλίου": 4,
    "Μαΐου": 5,
    "Μαϊου": 5,
    "Ιουνίου": 6,
    "Ιουλίου": 7,
    "Αυγούστου": 8,
    "Σεπτεμβρίου": 9,
    "Οκτωβρίου": 10,
    "Νοεμβρίου": 11,
    "Δεκεμβρίου": 12
}


def extract_date_from_text(text):

    pattern = re.compile(
        r"(\d{1,2})\s+"
        r"(Ιανουαρίου|Φεβρουαρίου|Μαρτίου|Απριλίου|"
        r"Μαΐου|Μαϊου|Ιουνίου|Ιουλίου|Αυγούστου|"
        r"Σεπτεμβρίου|Οκτωβρίου|Νοεμβρίου|Δεκεμβρίου)"
        r"\s+(\d{4})"
        r"(?:\s*·\s*(\d{1,2}):(\d{2}))?"
    )

    match = pattern.search(text)

    if not match:
        return None

    day = int(match.group(1))
    month = MONTHS[match.group(2)]
    year = int(match.group(3))

    hour = int(match.group(4)) if match.group(4) else 0
    minute = int(match.group(5)) if match.group(5) else 0

    return pd.Timestamp(
        year=year,
        month=month,
        day=day,
        hour=hour,
        minute=minute
    )


In [ ]:
def extract_full_text(soup):

    # Προσπαθούμε πρώτα να βρούμε το βασικό article
    article = soup.find("article")

    if article is None:

        article = soup.find(
            "div",
            class_=re.compile(
                r"(article|story|content|body)",
                re.I
            )
        )

    if article is None:
        article = soup

    paragraphs = []

    STOP_PHRASES = [
        "Διαβάστε επίσης",
        "Ακολουθήστε το Newsbomb",
        "Εγγραφείτε στο Newsletter",
        "Ροή Ειδήσεων",
        "Σχόλια"
    ]

    for p in article.find_all("p"):

        text = p.get_text(
            " ",
            strip=True
        )

        if not text:
            continue

        if len(text) < 20:
            continue

        if any(
            phrase.lower() in text.lower()
            for phrase in STOP_PHRASES
        ):
            break

        if text not in paragraphs:
            paragraphs.append(text)

    return "\n".join(paragraphs)


In [ ]:

def extract_article(session, url):

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        # ----------------
        # TITLE
        # ----------------

        title = None

        h1 = soup.find("h1")

        if h1:
            title = h1.get_text(
                " ",
                strip=True
            )

        if not title:

            meta_title = soup.find(
                "meta",
                property="og:title"
            )

            if meta_title:
                title = meta_title.get("content")


        # ----------------
        # DATE
        # ----------------

        page_text = soup.get_text(
            " ",
            strip=True
        )

        date = extract_date_from_text(
            page_text
        )


        # ----------------
        # AUTHOR
        # ----------------

        author = None

        meta_author = soup.find(
            "meta",
            attrs={"name": "author"}
        )

        if meta_author:
            author = meta_author.get("content")

        if not author:

            author_element = soup.find(
                class_=re.compile(
                    r"(author|writer)",
                    re.I
                )
            )

            if author_element:
                author = author_element.get_text(
                    " ",
                    strip=True
                )


        # ----------------
        # FULL TEXT
        # ----------------

        full_text = extract_full_text(
            soup
        )


        return {
            "site": SOURCE_NAME,
            "url": url,
            "title": title,
            "date": (
                date.date()
                if date is not None
                else None
            ),
            "datetime": date,
            "author": author,
            "full_text": full_text
        }

    except Exception as e:

        print(
            f"❌ Error στο άρθρο {url}: {e}"
        )

        return None



In [ ]:
def scrape_newsbomb():

    session = build_session()

    urls = collect_article_urls(
        session
    )

    articles = []

    print("\n📰 Κατέβασμα άρθρων...\n")

    for i, url in enumerate(urls, start=1):

        print(
            f"[{i}/{len(urls)}] {url}"
        )

        article = extract_article(
            session,
            url
        )

        if article is not None:

            # Κρατάμε ΜΟΝΟ άρθρα του 2026
            if (
                article["datetime"] is not None
                and article["datetime"].year == TARGET_YEAR
            ):

                articles.append(article)

        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )

    df = pd.DataFrame(articles)

    if not df.empty:

        df = df.drop_duplicates(
            subset=["url"]
        )

        df = df.sort_values(
            "datetime",
            ascending=False
        )

        df = df.reset_index(
            drop=True
        )

    return df

In [ ]:
newsbomb_2026_df = scrape_newsbomb()

print(
    "Συνολικά άρθρα Newsbomb 2026:",
    len(newsbomb_2026_df)
)

display(
    newsbomb_2026_df.head()
)

📄 Σελίδα 1: https://www.newsbomb.gr/technologia?page=1
📄 Σελίδα 2: https://www.newsbomb.gr/technologia?page=2
📄 Σελίδα 3: https://www.newsbomb.gr/technologia?page=3
📄 Σελίδα 4: https://www.newsbomb.gr/technologia?page=4
📄 Σελίδα 5: https://www.newsbomb.gr/technologia?page=5
📄 Σελίδα 6: https://www.newsbomb.gr/technologia?page=6
📄 Σελίδα 7: https://www.newsbomb.gr/technologia?page=7
📄 Σελίδα 8: https://www.newsbomb.gr/technologia?page=8
📄 Σελίδα 9: https://www.newsbomb.gr/technologia?page=9
📄 Σελίδα 10: https://www.newsbomb.gr/technologia?page=10
📄 Σελίδα 11: https://www.newsbomb.gr/technologia?page=11
📄 Σελίδα 12: https://www.newsbomb.gr/technologia?page=12
📄 Σελίδα 13: https://www.newsbomb.gr/technologia?page=13
📄 Σελίδα 14: https://www.newsbomb.gr/technologia?page=14
📄 Σελίδα 15: https://www.newsbomb.gr/technologia?page=15
📄 Σελίδα 16: https://www.newsbomb.gr/technologia?page=16
📄 Σελίδα 17: https://www.newsbomb.gr/technologia?page=17

🔗 Βρέθηκαν 425 μοναδικά URLs.

📰 Κατέβασμα άρθρω

,site,url,title,date,datetime,author,full_text
0,Newsbomb,https://www.newsbomb.gr/technologia/story/1764...,Έρχεται voucher για αγορά smartphone: Ποιοι δι...,2026-09-19,2026-09-19 16:45:00,Ζωή Μακρυγιάννη,Στο πρόγραμμα θα μπορούν να συμμετάσχουν οι πο...
1,Newsbomb,https://www.newsbomb.gr/technologia/story/1763...,Φορέσαμε το νέο Huawei Watch GT 7 Pro στο Μόνα...,2026-09-18,2026-09-18 14:03:00,Χρήστος Κάβουρας,Η Huawei έχει οραματιστεί την επόμενη μέρα της...
2,Newsbomb,https://www.newsbomb.gr/technologia/story/1763...,WSJ: Ξεχάστε την «Αποκάλυψη» της AI – Οι πραγμ...,2026-09-17,2026-09-17 07:40:00,Δημήτρης Μάνωλης,Μέσα από την αμερικανική βιομηχανία της AI υπο...
3,Newsbomb,https://www.newsbomb.gr/technologia/story/1763...,"«Δορυφόροι - κυνηγοί», «ρομποτικοί αρπάγες» κα...",2026-09-16,2026-09-16 07:24:00,Δημήτριος Παπαγεωργίου,Ο υπουργός Αεροπορίας των ΗΠΑ Τρόι Μέινκ επιβε...
4,Newsbomb,https://www.newsbomb.gr/technologia/story/1763...,Προειδοποίηση Μπιλ Γκέιτς: «Η τεχνητή νοημοσύν...,2026-09-15,2026-09-15 19:34:00,Δημήτριος Παπαγεωργίου,Ο συνιδρυτής της Microsoft προειδοποιεί από το...


In [ ]:
newsbomb_2026_df.info()

display(
    newsbomb_2026_df[
        [
            "title",
            "date",
            "author",
            "full_text"
        ]
    ].head(10)
)

print(
    "Κενά full_text:",
    newsbomb_2026_df["full_text"].isna().sum()
)

print(
    "Πρώτη ημερομηνία:",
    newsbomb_2026_df["date"].min()
)

print(
    "Τελευταία ημερομηνία:",
    newsbomb_2026_df["date"].max()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   site       399 non-null    object        
 1   url        399 non-null    object        
 2   title      399 non-null    object        
 3   date       399 non-null    object        
 4   datetime   399 non-null    datetime64[ns]
 5   author     399 non-null    object        
 6   full_text  399 non-null    object        
dtypes: datetime64[ns](1), object(6)
memory usage: 21.9+ KB


,title,date,author,full_text
0,Έρχεται voucher για αγορά smartphone: Ποιοι δι...,2026-09-19,Ζωή Μακρυγιάννη,Στο πρόγραμμα θα μπορούν να συμμετάσχουν οι πο...
1,Φορέσαμε το νέο Huawei Watch GT 7 Pro στο Μόνα...,2026-09-18,Χρήστος Κάβουρας,Η Huawei έχει οραματιστεί την επόμενη μέρα της...
2,WSJ: Ξεχάστε την «Αποκάλυψη» της AI – Οι πραγμ...,2026-09-17,Δημήτρης Μάνωλης,Μέσα από την αμερικανική βιομηχανία της AI υπο...
3,"«Δορυφόροι - κυνηγοί», «ρομποτικοί αρπάγες» κα...",2026-09-16,Δημήτριος Παπαγεωργίου,Ο υπουργός Αεροπορίας των ΗΠΑ Τρόι Μέινκ επιβε...
4,Προειδοποίηση Μπιλ Γκέιτς: «Η τεχνητή νοημοσύν...,2026-09-15,Δημήτριος Παπαγεωργίου,Ο συνιδρυτής της Microsoft προειδοποιεί από το...
5,Επιστήμονες βρήκαν έναν τρόπο να παράγουν καύσ...,2026-09-12,Μάνος Χατζηγιάννης,Οι επιστήμονες έχουν προτείνει την απόκτηση κα...
6,Ο Διευθύνων Σύμβουλος της Anthropic ζητά άμεση...,2026-09-12,Δέσποινα Σοφιανίδου,"Ο Αμοντέι, ισχυρός υποστηρικτής της προόδου τη..."
7,Η νέα γενιά προϊόντων Apple έρχεται στη Vodafone,2026-09-10,Newsbomb,Πότε ξεκινούν οι προπαραγγελίες για τα νέα iPh...
8,iPhone 18 Pro και iPhone Duo: Το πρώτο αναδιπλ...,2026-09-10,Newsbomb,Τα iPhone 18 Pro κυκλοφορούν στις 18 Σεπτεμβρί...
9,iPhone Duo: Ανάμεικτες οι πρώτες αντιδράσεις γ...,2026-09-10,Γιάννης Φιλιππάκος,Η Apple μπήκε για πρώτη φορά στην αγορά των fo...


Κενά full_text: 0
Πρώτη ημερομηνία: 2026-01-01
Τελευταία ημερομηνία: 2026-09-19


In [ ]:
import base64
import requests
from google.colab import userdata

def save_df_to_github(df, repo, path, token, branch="main", commit_message="Update dataset"):

    csv_content = df.to_csv(index=False)

    encoded_content = base64.b64encode(
        csv_content.encode("utf-8-sig")
    ).decode("utf-8")

    url = f"https://api.github.com/repos/{repo}/contents/{path}"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json"
    }

    # Έλεγχος αν υπάρχει ήδη το αρχείο
    existing = requests.get(
        url,
        headers=headers,
        params={"ref": branch}
    )

    data = {
        "message": commit_message,
        "content": encoded_content,
        "branch": branch
    }

    # Αν υπάρχει ήδη, χρειάζεται το sha για update
    if existing.status_code == 200:
        data["sha"] = existing.json()["sha"]

    response = requests.put(
        url,
        headers=headers,
        json=data
    )

    response.raise_for_status()

    print(f"✅ Αποθηκεύτηκε το {path} στο GitHub")

In [ ]:
save_df_to_github(
    newsbomb_2026_df,
    repo="annatsamoyra-prog/data-story-",
    path="newsbomb_technologia_2026.csv",
    token=userdata.get("newtoken")
)

✅ Αποθηκεύτηκε το newsbomb_technologia_2026.csv στο GitHub
